# 지도 학습 — 분류 : 로지스틱 회귀로 타이타닉 생존 여부 예측

수업 중 작성한 필기 `Data_Preprocessing03` · `로직스틱회귀실습` 과 개념 메모 `notbook03.py` 를 **실행 가능하도록 하나로 정리한 노트**입니다.
(교재 `머신러닝` p.47~56 — 11. 지도 학습을 이용한 분류)

메모에서 맞는 내용은 살리고, 틀리거나 헷갈리게 적힌 부분은 교재와 실행 결과로 확인해 고쳤습니다.
원본에서 고친 부분은 **📝 고친 부분** 으로 따로 적어 두었습니다.

> 실습 데이터 `titanic_train.csv` · `titanic_test.csv` 는 이 노트북과 **같은 폴더**에 있어야 합니다. 전처리 결과는 `titanic_cleaned.csv` 로 저장됩니다.
> `scikit-learn` · `plotly` 가 필요합니다 → `pip install scikit-learn plotly`

## 1. 로지스틱 회귀 — "이름은 회귀, 목적은 분류" (교재 p.48~52)

**선형 회귀 모델을 이용한 분류 알고리즘.** 선형 회귀처럼 직선(점수 z)을 계산하지만, 그 값을 그대로 쓰지 않고 **확률로 바꿔서 분류**한다.

```
학습 데이터 → ① 선형 회귀 : z = w1·x1 + w2·x2 + … + b
           → ② 시그모이드 : P = σ(z)  (0 ~ 1 사이 확률)
           → ③ 분류      : P ≥ 0.5 이면 1(True), 아니면 0(False)
           ↔ 정답(1/0) 과 비교 → 손실(loss)
```

### 왜 확률로 바꿀까? (p.49)

양수/음수로만 판단하면 "**A 클래스다**" 라는 결과만 알 수 있지만, 확률로 바꾸면 "**A 클래스일 확률이 95%다**" 라는 정보까지 얻는다.
예) 암 진단 결과가 51% 인 환자와 99% 인 환자 — 둘 다 "양성"이지만 의사는 다르게 대응할 수 있다.

### 시그모이드(Sigmoid) 함수

$$\sigma(z) = \frac{1}{1 + e^{-z}} \qquad (e \approx 2.718,\ \text{자연 상수})$$

**-∞ ~ +∞ 의 입력을 0 ~ 1 사이로 압축하는 S자 곡선.**

| 입력 z | 시그모이드 값 (확률 P) |
|--------|---------------------|
| -∞ 로 갈수록 | **0** 에 가까워진다 |
| **0** | **0.5** ← **결정 경계** |
| +∞ 로 갈수록 | **1** 에 가까워진다 |

- 시그모이드 값 = A 클래스(1)일 확률 **P** → B 클래스(0)일 확률은 **1 − P**
- 0 과 1 에 **가까워질 뿐 정확히 0, 1 이 되지는 않는다**

In [ ]:
import numpy as np

# 시그모이드 함수 직접 만들기
def sigmoid(z):
    return 1 / (1 + np.exp(-z))     # np.exp(-z) = e 의 -z 제곱

print("자연 상수 e =", round(np.e, 4))
print("-" * 40)

for z in [-10, -2, -0.5, 0, 0.5, 2, 10]:
    p = sigmoid(z)
    print(f"z = {z:>5} → 확률 P = {p:.4f} → 분류 : {int(p >= 0.5)}")

### 결정 경계 — 시그모이드 값이 0.5 가 되는 곳

σ(z) = 0.5 ⇔ **z = 0** 이다. 그래서 결정 경계는 **직선 z = 0** 이다.

특성이 2개(x, y)이고 학습 결과가 **z = −x + y − 1** 이라면 —

| 위치 | z | 확률 | 분류 |
|------|---|------|------|
| 직선 **y = x + 1** 위 | z = 0 | 0.5 | 경계 (애매) |
| 직선보다 **위쪽** (y > x + 1) | z > 0 | > 0.5 | **Yes (1)** |
| 직선보다 **아래쪽** (y < x + 1) | z < 0 | < 0.5 | **No (0)** |

In [ ]:
# 메모의 예 : z = -x + y - 1  →  결정 경계는 y = x + 1
def z_score(x, y):
    return -x + y - 1

for x, y in [(1, 2), (1, 4), (1, 0)]:          # 직선 위 / 위쪽 / 아래쪽
    z = z_score(x, y)
    print(f"점({x}, {y}) : z = {z:>2} → 확률 {sigmoid(z):.3f} → {'Yes' if sigmoid(z) >= 0.5 else 'No'}")

### 학습 과정 — 손실(loss) 최적화 (p.51)

**정답이 1 이면 예측 확률을 1 에 가깝게, 정답이 0 이면 0 에 가깝게** 만드는 최적의 직선 $w_1x_1 + w_2x_2 + \cdots + b$ 를 찾는다.

| 이름 | 종류 | 뜻 |
|------|------|-----|
| `max_iter` | 매개변수 (설정) | 학습(직선의 기울기 조정)을 **최대 몇 번 반복할지** — 기본값 100 |
| `coef_` | 속성 (결과) | 학습으로 찾은 최적 직선의 **기울기** $w_1, w_2, \ldots$ |
| `intercept_` | 속성 (결과) | 최적 직선의 **절편** $b$ |

| 장점 | 단점 |
|------|------|
| **매우 빠른** 학습 속도 | 결정 경계가 **직선**이라 비선형 데이터에 한계 |

> **📝 고친 부분** (`notbook03.py` 메모)
> - 결정 경계 · 시그모이드 · 자연 상수 e · "0 과 1 에 수렴" · "정답이 1 이면 1 에, 0 이면 0 에 가깝게" → **맞다.** 그대로 정리
> - "Y = X + 1, −X + Y − 1 = 0 이 결정 경계" → 맞는데, **z = −x + y − 1 일 때** 라는 조건이 빠져 있었다. z = 0 인 곳이 결정 경계다
> - "3. accuracy 계산 4. 정확도를 1 에 가깝게 만드는 게 중요" → 학습이 직접 줄이는 건 **손실(log loss)** 이다. 정확도(accuracy)는 학습이 끝난 뒤 성능을 재는 **평가 지표**
> - "**Max_irer**: 학습 횟수 후 → coef_ 최적의 **선영** 회귀 직선" → `max_iter` 는 **최대** 반복 횟수(그 전에 충분히 수렴하면 일찍 멈춘다), `coef_` 는 그 직선의 **기울기(가중치)**
> - "z : 1단계에서 계산한 점수" → z = $w_1x_1 + \cdots + b$, 즉 **선형 회귀 식의 결과값**

## 2. 전처리 복습 — `pd.concat()` 과 조건 비교

타이타닉 데이터는 학습용 · 평가용 파일 2개로 나뉘어 있어서, 전처리를 한 번에 하려고 **먼저 이어 붙인다.**

> `concat` 은 **같은 컬럼끼리 위아래로 이어 붙이기**, SQL 의 `JOIN` 과 같은 `pd.merge()` 는 **공통 키로 옆으로 짝 맞추기**다. 성질이 다르다.

In [ ]:
import pandas as pd

# 딕셔너리 → DataFrame 3개 (4행, 5행, 4행)
data1 = {'A': ['A0', 'A1', 'A2', 'A3'],
         'B': ['B0', 'B1', 'B2', 'B3'],
         'C': ['C0', 'C1', 'C2', 'C3'],
         'D': ['D0', 'D1', 'D2', 'D3']}

data2 = {'A': ['A4', 'A5', 'A6', 'A7', 'A8'],
         'B': ['B4', 'B5', 'B6', 'B7', 'B8'],
         'C': ['C4', 'C5', 'C6', 'C7', 'C8'],
         'D': ['D4', 'D5', 'D6', 'D7', 'D8']}

data3 = {'A': ['A9', 'A10', 'A11', 'A12'],
         'B': ['B9', 'B10', 'B11', 'B12'],
         'C': ['C9', 'C10', 'C11', 'C12'],
         'D': ['D9', 'D10', 'D11', 'D12']}

df1 = pd.DataFrame(data=data1)
df2 = pd.DataFrame(data=data2)
df3 = pd.DataFrame(data=data3)

# 같은 컬럼끼리 위아래로 병합 → 인덱스 0 ~ 12 새로 매김
df_concat = pd.concat([df1, df2, df3], ignore_index=True)
print(df_concat)

In [ ]:
# 조건 비교 : A 가 'A4' 이고(&) B 가 'B4' 인 행 → True / False 마스크
mask = (df_concat['A'] == 'A4') & (df_concat['B'] == 'B4')
print(mask.tolist())

print("-" * 40)

# 마스크로 행 추출 (True 인 행만)
print(df_concat[mask])

> **📝 고친 부분**
> - 주석 "SQL **marge**" → `merge`, 변수 이름 `marge` → `df_concat`
> - 결과 확인 `print` 를 `''' … '''` 문자열로 막아 둔 것 → 정리, `print("\tA\n")` 처럼 의미 없는 출력 제거
> - 마스크(True/False)만 찍고 끝나던 것 → 마스크로 **실제 행을 추출**하는 데까지

## 3. 타이타닉 데이터 (교재 p.53)

**1,309명** = 학습용(train) **891명** + 평가용(test) **418명**, 컬럼 12개.

| 컬럼 | 뜻 | 컬럼 | 뜻 |
|------|-----|------|-----|
| **`Survived`** | **생존 여부 (정답 : 1 생존, 0 사망)** | `Parch` | 함께 탄 부모/자녀 수 |
| `PassengerId` | 승객 번호 | `Ticket` | 티켓 번호 |
| `Pclass` | 객실 등급 (1 · 2 · 3등석) | `Fare` | 요금 |
| `Name` | 이름 | `Cabin` | 선실 이름 |
| `Gender` | 성별 (교재에서는 `Sex`) | `Embarked` | 승선 항구 (C 셰르부르, Q 퀸스타운, S 사우샘프턴) |
| `Age` | 나이 | `SibSp` | 함께 탄 형제자매/배우자 수 |

**목표** — ① 데이터 전처리 → ② 학습용으로 학습 후 평가용으로 생존 여부 분류 · 모델 평가

In [ ]:
# 파일 경로 (노트북과 같은 폴더)
train_path = './titanic_train.csv'
test_path = './titanic_test.csv'

# DataFrame 생성
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print("학습용 :", df_train.shape, "/ 평가용 :", df_test.shape)
df_train.head()

> **📝 고친 부분**
> - 절대 경로 `C:\Users\Jinho\work\…`, `'C:./titanic_train.csv'` (드라이브 이름과 상대 경로가 섞임) → `./titanic_train.csv`
> - `warnings.filterwarnings('ignore')` 로 **모든 경고를 끄던 셀** 제거 — 문제가 생겨도 원인을 놓친다
> - 891행 · 418행을 통째로 찍던 `display(df_train)` · `display(df_test)` → 모양과 `head()` 만

## 4. 데이터 전처리 (교재 p.54)

| 순서 | 작업 | 대상 |
|------|------|------|
| ① | 데이터 병합 | `pd.concat([df_train, df_test], ignore_index=True)` |
| ② | 컬럼별 누락 데이터 처리 | `Cabin` 삭제 · `Age` 중앙값 · `Embarked` 최빈값 · `Fare` 행 삭제 |
| (+) | 이상치 처리 | `Fare` — IQR 범위로 `clip` |
| ③ | 불필요한 컬럼 삭제 | `PassengerId` · `Name` · `Age` · `Ticket` |
| ④ | One-Hot encoding | `Gender` · `Embarked` |
| ⑤ | 결과 저장 | `titanic_cleaned.csv` |

### 4-1. 데이터 병합 · 누락 데이터 확인

In [ ]:
# ① 학습용 + 평가용 → 하나로 병합 (1,309행)
df = pd.concat([df_train, df_test], ignore_index=True)
print("병합 후 :", df.shape)

print("-" * 40)

# 각 컬럼별 누락 데이터 수
num_nulls = df.isnull().sum()
print(f"컬럼별 누락 데이터 수 : \n{num_nulls}")

| 컬럼 | 누락 | 처리 방법 | 이유 |
|------|------|----------|------|
| `Cabin` | 1,014 (77%) | **컬럼 삭제** | 대부분이 비어 있어 채울 근거가 없다 |
| `Age` | 263 | **중앙값**으로 채우기 | 숫자형 → 중앙값 또는 최빈값. 평균은 이상치에 끌려간다 |
| `Embarked` | 2 | **최빈값**(`S`)으로 채우기 | 범주형 → 가장 많은 값 |
| `Fare` | 1 | **행 삭제** | 1행뿐이라 지워도 영향이 거의 없다 |

### 4-2. `Cabin` 삭제 · `Age` 중앙값 대체 + 구간화

In [ ]:
# Cabin 컬럼 삭제 → drop(columns=[...])
df1 = df.drop(columns=['Cabin'])

# Age 컬럼 요약 통계량 → 중앙값(50%) 확인
print(f"Age 요약 통계량 : \n{df1['Age'].describe().round(2)}")

print("-" * 40)

# 중앙값으로 누락 대체
age_med = df1['Age'].median()
df1['Age'] = df1['Age'].fillna(age_med)
print(f"Age 중앙값 : {age_med} → 대체 후 누락 수 : {df1['Age'].isnull().sum()}")

In [ ]:
# Age 구간화 → 8단계 (bins=8 : 최솟값 ~ 최댓값을 같은 너비로 8등분)
labels = [0, 1, 2, 3, 4, 5, 6, 7]
df1['Age_step'] = pd.cut(x=df1['Age'], bins=8, labels=labels)

# 각 구간의 실제 범위 확인 → 약 10살 단위
print(pd.cut(df1['Age'], bins=8).cat.categories)

print("-" * 40)
print(df1[['Age', 'Age_step']].head())

### 4-3. `Embarked` 최빈값 대체 · `Fare` 누락 행 삭제

In [ ]:
# 승선 항구별 빈도수 / 비율
print(f"빈도수 : \n{df1['Embarked'].value_counts()}")
print("-" * 40)
print(f"비율 : \n{df1['Embarked'].value_counts(normalize=True).round(3)}")

print("-" * 40)

# 최빈값(빈도가 가장 큰 값)으로 채우기 → mode()[0] 으로 직접 구하면 'S' 를 손으로 적지 않아도 된다
embarked_mode = df1['Embarked'].mode()[0]
df1['Embarked'] = df1['Embarked'].fillna(embarked_mode)
print(f"Embarked 최빈값 : {embarked_mode} → 대체 후 누락 수 : {df1['Embarked'].isnull().sum()}")

In [ ]:
# Fare 누락 데이터(1행) 삭제 → dropna(subset=[...]), 인덱스 다시 매김
df2 = df1.dropna(subset=['Fare'], ignore_index=True)

print("삭제 후 :", df2.shape)
print("Fare 누락 수 :", df2['Fare'].isnull().sum())
print("전체 누락 수 :", df2.isnull().sum().sum())

> **📝 고친 부분** (`Data_Preprocessing03`)
> - `df_del = df_tot.drop(columns=['Cabin'])` 로 만들어 놓고 이후 작업은 계속 `df_tot` 에 해서 **`Cabin` 이 끝까지 남아** 원-핫 결과에 들어갔다 → 한 줄기(`df → df1 → df2 …`)로 이어지게 정리
> - `Fare02 = df_tot.dropna(...)` 결과를 쓰지 않아 **Fare 누락 1행이 그대로** 남았다 → 그대로 학습하면 `LogisticRegression` 이 NaN 때문에 에러
> - 변수 이름 `sum = …` → 파이썬 내장 함수 `sum()` 을 덮어써서 이후 `sum([...])` 이 고장 난다 → `num_nulls`
> - `age_count` 를 빈도수 · 요약 통계량 두 가지 뜻으로 재사용 → 바로 출력
> - 주석 "숫자값로 된", "숭선", "Embared" 오타
> - `Embarked` 를 `'S'` 로 손으로 적던 것 → `mode()[0]` 으로 최빈값을 구해 채움 (결과는 같다)

### 4-4. 이상치 처리 — `Fare` (IQR + `clip`)

Day 019 박스 플롯에서 배운 **IQR × 1.5** 기준으로 정상 범위를 정하고, 범위 밖 값은 **경계값으로 잘라 붙인다(`clip`).**

| 경계 | 식 |
|------|-----|
| 정상 범위 최솟값 | Q1 − 1.5 × IQR |
| 정상 범위 최댓값 | Q3 + 1.5 × IQR |

> 이상치를 **삭제**하면 그 승객의 다른 정보까지 잃는다. `clip` 은 행은 남기고 값만 경계로 바꾼다.

In [ ]:
import plotly.express as px

# 처리 전 Fare 분포 (박스 위로 점이 많이 찍힌다 = 이상치)
fig_fare1 = px.box(data_frame=df2, y='Fare', title='Fare — 이상치 처리 전')
fig_fare1.show()

In [ ]:
# Q1, Q3, IQR 계산
q1 = df2['Fare'].quantile(0.25)
q3 = df2['Fare'].quantile(0.75)
iqr = q3 - q1
print(f"Q1 = {q1:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}")

# 정상 범위 최솟값 / 최댓값
min_val = q1 - (1.5 * iqr)
max_val = q3 + (1.5 * iqr)
print(f"정상 범위 : {min_val:.2f} ~ {max_val:.2f}")
print("범위를 넘는 승객 수 :", (df2['Fare'] > max_val).sum())

# clip : 최솟값 미만 → min_val, 최댓값 초과 → max_val 로 일괄 대체
df2['Fare'] = df2['Fare'].clip(lower=min_val, upper=max_val)
print("처리 후 최댓값 :", round(df2['Fare'].max(), 2))

fig_fare2 = px.box(data_frame=df2, y='Fare', title='Fare — 이상치 처리 후')
fig_fare2.show()

> 요금은 음수가 없으니 최솟값(−27.17) 쪽은 걸리는 승객이 없고, 66.34 를 넘는 **171명**의 요금이 66.34 로 바뀌었다.

### 4-5. 불필요한 컬럼 삭제 · One-Hot encoding · 저장

In [ ]:
# ③ 불필요한 컬럼 삭제
#   PassengerId · Name · Ticket : 생존과 관계없는 식별용 값
#   Age : Age_step(구간화) 으로 대신 쓰므로 삭제
df3 = df2.drop(columns=['PassengerId', 'Name', 'Age', 'Ticket'])

# ④ One-Hot encoding : 문자열 특성 컬럼(Gender, Embarked) → 0/1 컬럼
df4 = pd.get_dummies(data=df3, columns=['Gender', 'Embarked'], dtype=int)

print(df4.shape)
df4.head()

In [ ]:
# ⑤ 전처리 완료 데이터 저장 (index=False : 행 번호는 저장하지 않음)
file_path = './titanic_cleaned.csv'
df4.to_csv(file_path, index=False)
print("저장 완료 :", file_path)

> **📝 고친 부분**
> - `pd.get_dummies(...)` 에 `dtype=int` 가 없어 결과가 `True/False` 로 나오던 것 → `dtype=int` 로 0/1
> - 주석 "string 를 정수로 해놔야 한다" → 모델은 **숫자만** 받으므로 문자열 컬럼(`Gender`, `Embarked`)을 원-핫 인코딩한다
> - 삭제 이유를 컬럼별로 주석에 남김 (`Age` 는 `Age_step` 으로 대체)

## 5. 학습용 · 평가용 데이터 생성 + 표준화 (교재 p.54 ⑥ ⑦)

In [ ]:
# 저장한 파일 다시 불러오기
df = pd.read_csv('./titanic_cleaned.csv')

# 특성(X) / 정답(y) 나누기
X_data = df.drop(columns=['Survived'])
y_data = df['Survived']
print("X_data :", X_data.shape, "/ y_data :", y_data.shape)

print("-" * 40)

# 정답의 비율 확인 → 균등한가?
print(f"정답 비율 : \n{y_data.value_counts(normalize=True).round(3)}")

> 사망(0) **62%** : 생존(1) **38%** — 고르지 않다. 그래서 나눌 때 `stratify=y_data` 로 **원본 비율을 학습용 · 평가용에 똑같이** 유지한다.

In [ ]:
from sklearn.model_selection import train_test_split

# 80 : 20 으로 나누기 (stratify : 정답 비율 유지)
X_train, X_test, y_train, y_test = train_test_split(
    X_data,
    y_data,
    test_size=0.2,
    random_state=0,
    stratify=y_data
)

print("학습용 :", X_train.shape, "/ 평가용 :", X_test.shape)
print("X_train · y_train 짝이 맞는가? :", (X_train.index == y_train.index).all())

print("-" * 40)

# 학습용 · 평가용 정답 비율 → 원본(0.622 : 0.378)과 거의 같다
print("학습용 :", y_train.value_counts(normalize=True).round(3).tolist())
print("평가용 :", y_test.value_counts(normalize=True).round(3).tolist())

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# 학습용 → fit_transform : 학습용의 평균 · 표준편차를 계산해서 적용
# 평가용 → transform     : 학습용의 평균 · 표준편차로 변환만
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. `LogisticRegression` 모델 (교재 p.55)

```python
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

logistic = LogisticRegression()                 # ① 모델 생성
logistic.fit(X_train, y_train)                  # ② 학습
pred_test = logistic.predict(X_test)            # ③ 평가용 데이터로 예측 (0 / 1)
acc_lr = accuracy_score(y_test, pred_test)      # ④ 정확도 = 맞힌 개수 / 전체 개수
```

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# ① 모델 생성 (max_iter : 기울기 조정 최대 반복 횟수, 기본값 100)
logistic = LogisticRegression(max_iter=100)

# ② 학습 : 정답 1 → 확률 1 에 가깝게, 정답 0 → 0 에 가깝게 만드는 직선 찾기
logistic.fit(X_train_scaled, y_train)
print("학습 완료! 실제 반복 횟수 :", logistic.n_iter_[0], "번")

In [ ]:
# 학습 결과 : 최적 직선 z = w1·x1 + … + w10·x10 + b
coef = pd.Series(logistic.coef_[0], index=X_data.columns)   # coef_ 는 2차원 → [0] 으로 꺼냄
print("기울기(가중치) coef_ :")
print(coef.round(3).sort_values())

print("-" * 40)
print("절편 intercept_ :", logistic.intercept_.round(3))

> **가중치 읽기** — 양수면 생존 확률을 **올리고**, 음수면 **내린다.**
> - `Gender_female` **+0.88** / `Gender_male` **−0.88** → 여성일수록 생존 (실제 생존율 여성 83% · 남성 13%)
> - `Pclass` **−0.69** → 등급 숫자가 클수록(3등석) 생존 확률이 낮다
> - `Gender_female` 과 `Gender_male` 이 정확히 **부호만 반대** — 한쪽이 1 이면 다른 쪽은 무조건 0 이라 같은 정보를 두 번 담고 있기 때문이다. (`get_dummies(..., drop_first=True)` 로 하나만 남길 수 있다)

In [ ]:
# ③ 평가용 데이터로 예측
pred_test = logistic.predict(X_test_scaled)        # 결과 : 0 / 1
proba_test = logistic.predict_proba(X_test_scaled) # 결과 : [0 일 확률, 1 일 확률]

result = pd.DataFrame({
    '사망 확률(0)': proba_test[:, 0].round(3),
    '생존 확률(1)': proba_test[:, 1].round(3),
    '예측': pred_test,
    '정답': y_test.values,
})
result.head(10)

In [ ]:
# predict_proba 는 정말 시그모이드일까? → z(직선 점수)를 직접 시그모이드에 넣어 비교
z = logistic.decision_function(X_test_scaled)     # z = w·x + b
print("z 값       :", z[:5].round(3))
print("σ(z)       :", sigmoid(z[:5]).round(3))
print("생존 확률  :", proba_test[:5, 1].round(3))
print("같은가? :", np.allclose(sigmoid(z), proba_test[:, 1]))

In [ ]:
# ④ 모델 평가 : 정확도(accuracy) = 맞힌 개수 / 전체 개수
acc_lr = accuracy_score(y_test, pred_test)

correct = (pred_test == y_test).sum()
print(f"평가용 {len(y_test)}명 중 {correct}명 맞힘 → 정확도 {acc_lr:.4f} ({acc_lr * 100:.1f}%)")

print("-" * 40)

# 비교 기준 : 모두 '사망(0)' 이라고 찍기만 해도 맞는 비율
baseline = (y_test == 0).mean()
print(f"전부 0 으로 찍었을 때 : {baseline * 100:.1f}%")

# 틀린 유형
print("살았는데 사망으로 예측 :", ((pred_test == 0) & (y_test == 1)).sum(), "명")
print("사망했는데 생존으로 예측 :", ((pred_test == 1) & (y_test == 0)).sum(), "명")

> **결과 해석**
> - 평가용 **262명 중 225명**을 맞혀 정확도 **85.9%**
> - 아무것도 학습하지 않고 "전부 사망" 이라고 찍어도 **62.2%** 는 맞는다. 정확도는 항상 이런 **기준선과 비교**해야 의미가 있다
> - 틀린 37명 중 21명은 **살았는데 사망으로**, 16명은 **사망했는데 생존으로** 예측했다

> **📝 고친 부분** (`로직스틱회귀실습`)
> - 학습 결과 셀 주석 "생성된 직선의 기울기" → `coef_` 를 특성 이름과 짝지어 출력하고 부호 해석 추가
> - "모델 **평간**" → 평가, 주석 `#85.8%` → 실제 값 **0.8588 (85.9%)**
> - `display(acc_lr)` → 맞힌 개수와 함께 문장으로 출력
> - 교재 p.49 "확률 정보를 얻을 수 있다" 를 직접 확인하는 `predict_proba()` · 시그모이드 검증 코드 추가
> - 메모 실습 목표 "전체에서 몇 개를 맞추었나" → 맞다. 그게 바로 **정확도**

## 7. 오늘 정리

| 단계 | 코드 |
|------|------|
| 병합 | `pd.concat([df_train, df_test], ignore_index=True)` |
| 누락 처리 | `drop(columns=['Cabin'])` · `fillna(median())` · `fillna(mode()[0])` · `dropna(subset=['Fare'])` |
| 구간화 | `pd.cut(df['Age'], bins=8, labels=[0, …, 7])` |
| 이상치 | `clip(lower=Q1 − 1.5·IQR, upper=Q3 + 1.5·IQR)` |
| 인코딩 | `pd.get_dummies(df, columns=['Gender', 'Embarked'], dtype=int)` |
| 분할 | `train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)` |
| 표준화 | `fit_transform(X_train)` / `transform(X_test)` |
| 학습 · 예측 | `LogisticRegression().fit(...)` → `predict()` · `predict_proba()` |
| 평가 | `accuracy_score(y_test, pred_test)` → **0.859** |

- **로지스틱 회귀** = 선형 회귀 점수 z → **시그모이드** → 0 ~ 1 확률 → 0.5 기준으로 분류. 이름은 회귀, 목적은 **분류**.
- **결정 경계**는 확률 0.5 = **z = 0** 인 직선.
- 학습은 **손실 최소화**(정답 1 → 확률 1, 정답 0 → 확률 0), 정확도는 **평가 지표**.
- `max_iter` = 최대 반복 횟수, `coef_` = 최적 직선의 기울기.
- 정확도는 "전부 한쪽으로 찍기" **기준선**(62.2%)과 비교해서 읽는다.